# 품질 RF + AF 통합 v2 (최종본)

Git에 포함된 **학습 모델을 바로 불러오는** 팀원용 노트북입니다. Drive 연결·재학습·학습 데이터 다운로드가 필요 없습니다.
런타임 유형은 T4 GPU로 선택하세요. 이 최종본은 연구용 실행본이며 새 센서/임상 정확도 검증은 완료되지 않았습니다.


In [ ]:
from pathlib import Path
import subprocess, sys
repo=Path('/content/capdyssey_final')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn==1.6.1','wfdb','h5py','joblib'],check=True)
import torch
assert torch.cuda.is_available(), 'T4 GPU를 선택하세요.'
MODEL=repo/'models/quality_af_v2_final'
assert (MODEL/'quality_rf.joblib').exists(), '학습 모델이 포함된 최신 저장소가 필요합니다.'
sys.path.insert(0,str(MODEL/'scripts'))
from quality_af_v2 import QualityAF
predictor=QualityAF(MODEL)
print('학습된 최종본 로드 완료. 재학습 없음.')
print('품질 기준:',predictor.quality_threshold,'AF 기준:',predictor.af.threshold)


## 실제 PPG 입력

단일 채널 원본 PPG 10초와 실제 샘플링 주파수가 필요합니다. 이미 정규화한 값이나 ECG 정답을 입력하지 않습니다.
`status=remeasure`는 비AF가 아닙니다. AF/비AF 판정이 없으며 재측정이 필요하다는 뜻입니다.


In [ ]:
def predict_ppg(raw_ppg, sampling_hz):
    return predictor.predict(raw_ppg,sampling_hz)

# 센서 파형이 준비되면 실행:
# result=predict_ppg(my_ppg_array,100)
# print(result)


In [ ]:
import pandas as pd
display(pd.read_csv(MODEL/'development_test.csv'))
print('위 수치는 반복 사용한 시험군의 개발 평가입니다. 새 센서 성능이 아닙니다.')
